# __Цифровые инструменты финансиста__

## __День 2.__ Работа с данными

### __4. Бонус: как загрузить и обработать нестандартный формат__

#### 4.1. Загрузим данные для начала и распознаем их

Возьмем годовой отчет [одного крупного российского банка](https://www.vtb.ru/ir/statements/annual/) в формате pdf, загрузим его и распознаем при помощи OCR (optical character recognition) библиотеки [Tesseract](https://github.com/tesseract-ocr/tesseract). Эта библиотека уже есть у нас на платформе, поэтому нужно всего лишь скорректировать наш запрос:

```prompt

## Входные данные
- URL ссылка на файл для загрузки `https://www.vtb.ru/media-files/vtb.ru/sitepages/ir/statements/annual/rus_vtb_go_02072025_2024.pdf`
- Формат файла: `pdf`

## Задача
- написать код на Python для загрузки файла по ссылке и сохранить на диск
- открыть его и распознать pdf в текст со 119 по 120 страницы
- полученный текст сохранить в переменную `extracted_text`

## Технические ограничения
- Использовать библиотеки, приведенные ниже
- Код запускается в интерактивном ноутбуке Jupyter

## Технические требования
- Вывод данных: адаптированный под интерактивные ноутбуки, где возможно используй display вместо print 
- Стиль кода: PEP 8, black (длина строки 79)
- Документация: Docstrings в стиле Google

### Библиотеки
import pytesseract
from pdf2image import convert_from_path
from PIL import Image
```


In [1]:
import os
import requests
from pdf2image import convert_from_path
from PIL import Image
import pytesseract
from IPython.display import display, Markdown


def download_pdf_with_headers(url: str, save_path: str) -> str:
    """Скачивает PDF файл с поддельными заголовками браузера.
    
    Args:
        url: URL ссылка на PDF файл
        save_path: Путь для сохранения файла
        
    Returns:
        Путь к сохраненному файлу
    """
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
        'Accept': 'application/pdf,text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
        'Accept-Language': 'ru-RU,ru;q=0.8,en-US;q=0.5,en;q=0.3',
        'Accept-Encoding': 'gzip, deflate, br',
        'Connection': 'keep-alive',
        'Upgrade-Insecure-Requests': '1'
    }
    
    session = requests.Session()
    response = session.get(url, headers=headers, stream=True)
    response.raise_for_status()
    
    with open(save_path, 'wb') as file:
        for chunk in response.iter_content(chunk_size=8192):
            file.write(chunk)
    
    return save_path


def extract_pages_text(pdf_path: str, start_page: int, end_page: int) -> str:
    """Извлекает текст с указанных страниц PDF файла с помощью OCR.
    
    Args:
        pdf_path: Путь к PDF файлу
        start_page: Номер начальной страницы (включительно)
        end_page: Номер конечной страницы (включительно)
        
    Returns:
        Извлеченный текст с указанных страниц
    """
    # Конвертируем PDF в изображения
    images = convert_from_path(
        pdf_path,
        first_page=start_page,
        last_page=end_page,
        dpi=300
    )
    
    extracted_text = []
    
    for i, image in enumerate(images, start=start_page):
        # Применяем OCR к изображению
        text = pytesseract.image_to_string(image, lang='rus+eng')
        extracted_text.append(f"--- Страница {i} ---\n{text}")
    
    return '\n\n'.join(extracted_text)


# Основной код
pdf_url = "https://www.vtb.ru/media-files/vtb.ru/sitepages/ir/statements/annual/rus_vtb_go_02072025_2024.pdf"
pdf_filename = "vtb_report_2024.pdf"

try:
    # Скачиваем PDF файл с заголовками
    print("Загрузка PDF файла (с эмуляцией браузера)...")
    download_pdf_with_headers(pdf_url, pdf_filename)
    print(f"✓ Файл сохранен как: {pdf_filename}")
    
    # Извлекаем текст со страниц 119 по 120
    print("\nИзвлечение текста со страниц 119-120...")
    extracted_text = extract_pages_text(pdf_filename, 119, 120)
    
    # Выводим результат
    display(Markdown("### Извлеченный текст со страниц 119-120"))
    display(Markdown(f"```\n{extracted_text}\n```"))
    
    # Также сохраняем в файл для проверки
    with open("extracted_text.txt", "w", encoding="utf-8") as f:
        f.write(extracted_text)
    print("\n✓ Текст также сохранен в файл 'extracted_text.txt'")
    
except requests.exceptions.HTTPError as e:
    print(f"❌ Ошибка HTTP: {e}")
    print("\nВозможные решения:")
    print("1. Проверьте, открывается ли ссылка в обычном браузере")
    print("2. Возможно, файл требует авторизации или перемещен")
    print("3. Попробуйте скачать файл вручную и указать локальный путь")
    
    # Альтернативный вариант - работа с локальным файлом
    print("\n--- Альтернативный вариант ---")
    local_pdf = input("Если вы скачали PDF вручную, укажите путь к файлу: ")
    if os.path.exists(local_pdf):
        print("Извлекаем текст из локального файла...")
        extracted_text = extract_pages_text(local_pdf, 119, 120)
        display(Markdown("### Извлеченный текст со страниц 119-120"))
        display(Markdown(f"```\n{extracted_text}\n```"))
    else:
        print("Файл не найден. Пожалуйста, скачайте PDF вручную с сайта ВТБ.")

Загрузка PDF файла (с эмуляцией браузера)...
✓ Файл сохранен как: vtb_report_2024.pdf

Извлечение текста со страниц 119-120...


### Извлеченный текст со страниц 119-120

```
--- Страница 119 ---
Годовой отчет Отчет Корпоративное Устойчивое Финансовая Приложения = ВТБ
2024 менеджмента управление развитие отчетность
4.1. Финансовая отчетность по МСФО
Ниже приведены ключевые финансовые показатели группы ВТБ по МСФО. Для лучшего понимания финансового
положения Группы, результатов ее деятельности и движений ее денежных средств данные ключевые финансовые показатели
должны рассматриваться совместно с проаудированной консолидированной финансовой отчетностью Группы.
обобщенная консолидированная финансовая отчетность за год, закончившийся 31 декабря 2024 г.,
доступна по ссылке: https://www.vtb.ru/ir/statements/results
Выдержки из обобщенного консолидированного отчета о прибылях и убытках
за год, закончившийся 31 декабря (в миллиардах российских рублей)
2024 г. 2023 г. Изменение 2024 г. 2023 г. Изменение
Процентные доходы, рассчитанные по методу эффективной 3979,4 2556,2 68,9% Доходы за вычетом расходов по операциям с иностранной валютой 50,7 152,9 -61,9%
процентной ставки и драгоценными металлами
Прочие процентные доходы 122,5 98,2 24,7% Доходы от досрочного расторжения обязательств и иные 28,5 19,0 48,9%
доходы за вычетом расходов, отличные от процентных, по
Процентные расходы (3 555,1) (1648,2) 115,7% операциям с финансовыми инструментами, учитываемыми по
амортизированной стоимости
Платежи в рамках системы страхования вкладов (59,6) (44,8) 55,0%
Доля в прибыли ассоциированных компаний и совместных предприятий 13,7 29,8 -54,0%
Чистые процентные доходы 487,2 761,4 -36,0%
Обесценение инвестиций в ассоциированные компании (0,5) (7,5) -93,3%
Создание резерва под кредитные убытки по долговым финансовым (25,9) (175,6) -86,4% и совместные предприятия
активам
Доходы от выбытия дочерних и ассоциированных компаний 64,0 3,0 2 033,3%
Чистые процентные доходы после создания резерва под 465,35 585,8 -20,9% и совместных предприятий
ожидаемые кредитные убытки
о Создание резерва под кредитные убытки по прочим финансовым (1,5) (11,9) -87,4%
Чистые комиссионные доходы 269,0 217,0 24,0% активам и обязательствам кредитного характера
Доходы за вычетом расходов / (расходы за вычетом доходов) по 214,2 (45,4) 571,8% Восстановление резерва под судебные иски и прочие обязательства 53 0,5 1666,7%
операциям с финансовыми инструментами, переоцениваемыми
по справедливой стоимости через прибыль или убыток, или через
Выручка и прочие доходы от операционной аренды оборудования 10,9 14,8 -26,4%

прочий совокупный доход

119


--- Страница 120 ---
—
Годовой отчет Отчет Корпоративное Устойчивое Финансовая Приложения — ВТБ
2024 менеджмента управление развитие отчетность
2024 г. 2023 г. Изменение 2024 г. 2023 г. Изменение
Расходы, связанные с оборудованием, сданным в аренду (13,3) (7,7) 72,7% Обесценение гудвила (0,5) (1,7) -70,6%
Результат инвестирования активов негосударственных пенсионных фондов 88,9 64,0 38,9% Обесценение земли, зданий и нематериальных активов, (22,0) (6,3) 249,2%
за исключением гудвила
Финансовые расходы страховых компаний и негосударственных (81,6) (67,2) 21,4%
пенсионных фондов Чистый убыток от выбытия основных средств и нематериальных (7,0) (16,2) -56,8%
активов, за исключением гудвила
Выручка по страхованию и перестрахованию от деятельности 156,0 120,8 291%
страховых компаний и негосударственных пенсионных фондов Прочие операционные расходы (75,5) (43,5) 69,0%
Расходы по страхованию и перестрахованию OT деятельности (152,3) (114,8) 15,2% Расходы на содержание персонала и административные расходы (478,8) (400,3) 19,6%
страховых компаний и негосударственных пенсионных фондов
7 Непроцентные расходы (581,8) (468,0) 24,3%
Превышение справедливой стоимости приобретенных чистых - 35,4 -100,0%
активов над затратами
д р Прибыль до налогообложения 575,1 518,2 11,0%
Прочие операционные доходы 15,5 11,5 55,4%
Расход по налогу на прибыль (22,5) (86,0) -73,8%
Прочие непроцентные доходы OT финансовой деятельности 4181 176,8 136,5%
Чистая прибыль после налогообложения 552,6 432,2 27,9%
Выручка и прочие доходы от прочей нефинансовой деятельности 51,5 35,5 45.1%
Убыток после налогообложения, полученный от дочерних компаний, (1,2) - н.п.
Себестоимость и прочие расходы по прочей нефинансовой (51,1) (29,8) 71,5% приобретенных исключительно для перепродажи
деятельности
Чистая прибыль 551,4 432,2 27,6%
Расходы от уценки недвижимости, предназначенной для продажи (0,1) (0,2) -50,0%
в ходе обычной деятельности Чистая прибыль, приходящаяся на:
Обесценение земли, зданий и нематериальных активов, за исключе- (0,4) - н.п. Акционеров материнского банка 555,3 420,6 27.5%
нием гудвила, используемых в прочей нефинансовой деятельности
о
Чистый убыток от выбытия основных средств и нематериальных (0,4) (0,4) 0,0% Неконтрольные доли участия 161 11.6 588%
активов, за исключением гудвила, используемых в прочей
нефинансовой деятельности Прибыль на акцию: базовая и с учетом разводнения (в российских 94,0 95,5 -1,6%
рублях Ha одну акцию) после консолидации
Чистая прибыль от изменения справедливой стоимости 7,0 1,5 366,7%
инвестиционной недвижимости, отраженный по результатам Прибыль на акцию: базовая и с учетом разводнения до убытка 94,2 95,5 -1,4%
переоценки или выбытия после налогообложения, полученного от дочерних компаний,
приобретенных исключительно для перепродажи (в российских
Выручка за вычетом расходов по прочей нефинансовой деятельности 6,5 6,6 -1,5% рублях на одну акцию) после консолидации

120

```


✓ Текст также сохранен в файл 'extracted_text.txt'


#### 4.2. Постобработка данных

Одним из самых простых вариантов будет обработка через ИИ, просто отправим нужный промпт, указав в нем текст для обработки и извлечения из него данных:

```prompt
Преобразуй текст ниже в формат json для удобного извлечения финансовых показателей отчетности

---
Текст:

<ВСТАВЬТЕ СЮДА НУЖНЫЙ ТЕКСТ>
```

...потом можно добавить еще и промпт для перекладки полученной json структуры в pandas датафрейм:

```prompt
...а теперь напиши код на Python, чтобы загрузить эти данные в pandas датафрейм
```

In [2]:
# Ваш JSON-данные (как строка или загруженные из файла)
json_data = '''
{
  "company": "ВТБ",
  "report_type": "Годовой отчет",
  "year": 2024,
  "standards": "МСФО",
  "financial_statements": {
    "comprehensive_income_statement": {
      "as_of_date": "31 декабря 2024",
      "currency": "миллиардах российских рублей",
      "figures": [
        {
          "indicator": "Процентные доходы, рассчитанные по методу эффективной процентной ставки",
          "2024": 3979.4,
          "2023": 2556.2,
          "change_percent": 68.9
        },
        {
          "indicator": "Прочие процентные доходы",
          "2024": 122.5,
          "2023": 98.2,
          "change_percent": 24.7
        },
        {
          "indicator": "Процентные расходы",
          "2024": -3555.1,
          "2023": -1648.2,
          "change_percent": 115.7
        },
        {
          "indicator": "Платежи в рамках системы страхования вкладов",
          "2024": -59.6,
          "2023": -44.8,
          "change_percent": 55.0
        },
        {
          "indicator": "Чистые процентные доходы",
          "2024": 487.2,
          "2023": 761.4,
          "change_percent": -36.0
        },
        {
          "indicator": "Создание резерва под кредитные убытки по долговым финансовым активам",
          "2024": -25.9,
          "2023": -175.6,
          "change_percent": -86.4
        },
        {
          "indicator": "Чистые процентные доходы после создания резерва под ожидаемые кредитные убытки",
          "2024": 465.35,
          "2023": 585.8,
          "change_percent": -20.9
        },
        {
          "indicator": "Чистые комиссионные доходы",
          "2024": 269.0,
          "2023": 217.0,
          "change_percent": 24.0
        },
        {
          "indicator": "Доходы за вычетом расходов по операциям с финансовыми инструментами, переоцениваемыми по справедливой стоимости через прибыль или убыток",
          "2024": 214.2,
          "2023": -45.4,
          "change_percent": 571.8
        },
        {
          "indicator": "Доходы за вычетом расходов по операциям с иностранной валютой и драгоценными металлами",
          "2024": 50.7,
          "2023": 152.9,
          "change_percent": -61.9
        },
        {
          "indicator": "Доходы от досрочного расторжения обязательств и иные доходы за вычетом расходов, отличные от процентных, по операциям с финансовыми инструментами, учитываемыми по амортизированной стоимости",
          "2024": 28.5,
          "2023": 19.0,
          "change_percent": 48.9
        },
        {
          "indicator": "Доля в прибыли ассоциированных компаний и совместных предприятий",
          "2024": 13.7,
          "2023": 29.8,
          "change_percent": -54.0
        },
        {
          "indicator": "Обесценение инвестиций в ассоциированные компании и совместные предприятия",
          "2024": -0.5,
          "2023": -7.5,
          "change_percent": -93.3
        },
        {
          "indicator": "Доходы от выбытия дочерних и ассоциированных компаний и совместных предприятий",
          "2024": 64.0,
          "2023": 3.0,
          "change_percent": 2033.3
        },
        {
          "indicator": "Создание резерва под кредитные убытки по прочим финансовым активам и обязательствам кредитного характера",
          "2024": -1.5,
          "2023": -11.9,
          "change_percent": -87.4
        },
        {
          "indicator": "Восстановление резерва под судебные иски и прочие обязательства",
          "2024": 53.0,
          "2023": 0.5,
          "change_percent": 1666.7
        },
        {
          "indicator": "Выручка и прочие доходы от операционной аренды оборудования",
          "2024": 10.9,
          "2023": 14.8,
          "change_percent": -26.4
        }
      ]
    }
  },
  "link_to_audited_financials": "https://www.vtb.ru/ir/statements/results"
}
'''

In [3]:
import pandas as pd
import json

# Способ 1: Загрузка из JSON строки
data = json.loads(json_data)

# Извлечение данных о финансовых показателях
figures = data['financial_statements']['comprehensive_income_statement']['figures']

# Создание DataFrame
df = pd.DataFrame(figures)

# Добавление мета-информации как атрибутов (опционально)
df.attrs['company'] = data['company']
df.attrs['year'] = data['year']
df.attrs['standards'] = data['standards']
df.attrs['currency'] = data['financial_statements']['comprehensive_income_statement']['currency']
df.attrs['as_of_date'] = data['financial_statements']['comprehensive_income_statement']['as_of_date']

print("DataFrame с финансовыми показателями:")
print(df)
print("\nМета-информация:")
print(f"Компания: {df.attrs['company']}")
print(f"Год: {df.attrs['year']}")
print(f"Стандарт: {df.attrs['standards']}")
print(f"Валюта: {df.attrs['currency']}")

DataFrame с финансовыми показателями:
                                            indicator     2024    2023  \
0   Процентные доходы, рассчитанные по методу эффе...  3979.40  2556.2   
1                            Прочие процентные доходы   122.50    98.2   
2                                  Процентные расходы -3555.10 -1648.2   
3        Платежи в рамках системы страхования вкладов   -59.60   -44.8   
4                            Чистые процентные доходы   487.20   761.4   
5   Создание резерва под кредитные убытки по долго...   -25.90  -175.6   
6   Чистые процентные доходы после создания резерв...   465.35   585.8   
7                          Чистые комиссионные доходы   269.00   217.0   
8   Доходы за вычетом расходов по операциям с фина...   214.20   -45.4   
9   Доходы за вычетом расходов по операциям с инос...    50.70   152.9   
10  Доходы от досрочного расторжения обязательств ...    28.50    19.0   
11  Доля в прибыли ассоциированных компаний и совм...    13.70    29.8   


#### 4.3. Постобработка данных "по-взрослому"

Копировать текст в ИИ и обратно - это не совсем технологично, было бы неплохо делать это бесшовно в коде. К счастью, у всех современных ИИ ассистентов есть [API (Application Programming Interface)](https://habr.com/ru/articles/780008/), которым мы и воспользуемся.

В этом разделе не будет промптинга, просто воспользуемся уже готовым кодом, заранее разработанным для демонстрации того, как можно работать с ИИ моделями через API.

In [4]:
import io
import re
import json
import pandas as pd
from typing import Tuple, Optional, Any

from callllm.yandexgpt_client import YandexGPTClient, json_data

In [5]:
# Инструкция для ИИ-ассистента задаст ему нужную роль
# и расскажет что делать с данными, которыми мы ему дадим
instruction = """
Ты — ИИ-ассистент для анализа финансовых данных.

Твоя задача извлекать из текста финансовые показатели и структурировать
их в JSON формат для дальнейшей загрузки в базу данных или pandas датафрейм

Получи текст от пользователя и верни данные **строго** в JSON формате. Не
добавляй никаких символов в начале или в конце JSON структуры. Если есть пустые 
значения, то укажи в них пустые строки (для текстовых полей) или нули (для 
показателей).

Если проблему невозможно решить с предоставленными данными, то верни
пустой JSON в виде `{}`.
"""

# Загрузим ключи доступа к API языковых моделей Яндекса
access_data = json_data('/home/jovyan/__DATA/.accessyagpt')

# Создаем ИИ-ассистента, а так как мы пишем на Python,
# то ИИ-ассистент будет жить в переменной yaclient
yaclient = YandexGPTClient(
    folder_id=access_data["folder_id"],
    api_key=access_data["api_key"],
    instruction_text=instruction
)

YandexGPTClient initialized successfully


In [6]:
# Наш промпт будет очень простой = задание + текст для обработки
# текст уже находится в переменной `extracted_text`
prompt = f"""
Возьми текст и извлеки из него данные

---
ТЕКСТ: 

{extracted_text}

"""

# Спросим нашего ИИ-ассистента, просто дадим ему наш промпт,
# а также еще ряд параметров - имя модели, длину контекста, температуру.
# Ответ сохраним в переменную `response`, потом с ней разберемся
response = yaclient.call_yandexgpt(
    prompt=prompt, 
    model_name="yandexgpt", 
    max_tokens=4000, 
    temperature=0.1
)

In [7]:
response

'```\n{\n  "Процентные доходы, рассчитанные по методу эффективной процентной ставки": {\n    "2024": 3979.4,\n    "2023": 2556.2,\n    "Изменение": 68.9\n  },\n  "Доходы за вычетом расходов по операциям с иностранной валютой и драгоценными металлами": {\n    "2024": 50.7,\n    "2023": 152.9,\n    "Изменение": -61.9\n  },\n  "Прочие процентные доходы": {\n    "2024": 122.5,\n    "2023": 98.2,\n    "Изменение": 24.7\n  },\n  "Доходы от досрочного расторжения обязательств и иные доходы за вычетом расходов": {\n    "2024": 28.5,\n    "2023": 19.0,\n    "Изменение": 48.9\n  },\n  "Процентные расходы": {\n    "2024": -3555.1,\n    "2023": -1648.2,\n    "Изменение": 115.7\n  },\n  "Платежи в рамках системы страхования вкладов": {\n    "2024": -59.6,\n    "2023": -44.8,\n    "Изменение": 55.0\n  },\n  "Доля в прибыли ассоциированных компаний и совместных предприятий": {\n    "2024": 13.7,\n    "2023": 29.8,\n    "Изменение": -54.0\n  },\n  "Чистые процентные доходы": {\n    "2024": 487.2,\n   

In [8]:
# Обработаем полученные результаты - 
# конвертируем его из текста в json,
# а потом в pandas датафрейм
json_data = json.loads(response.replace("`", ""))
df = pd.DataFrame(json_data).T

In [9]:
df.head()

,2024,2023,Изменение
"Процентные доходы, рассчитанные по методу эффективной процентной ставки",3979.4,2556.2,68.9
Доходы за вычетом расходов по операциям с иностранной валютой и драгоценными металлами,50.7,152.9,-61.9
Прочие процентные доходы,122.5,98.2,24.7
Доходы от досрочного расторжения обязательств и иные доходы за вычетом расходов,28.5,19.0,48.9
Процентные расходы,-3555.1,-1648.2,115.7
